In [ ]:
!pip install 'lerobot[pi0]' 'lerobot[pi]' 'lerobot[async]' 'lerobot[dataset]' pyngrok -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 154.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.4/340.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

In [ ]:
!find / -name "modeling_pi05.py" 2>/dev/null | head -1

/usr/local/lib/python3.13/dist-packages/lerobot/policies/pi05/modeling_pi05.py


In [ ]:
from pyngrok import ngrok
import subprocess, threading, os
from google.colab import userdata

# Apply vision tower patch (v2 direction: checkpoint saves vision_tower.{sub}.* without vision_model)
path = subprocess.run(["find", "/usr", "-name", "modeling_pi05.py"],
                      capture_output=True, text=True).stdout.strip().split("\n")[0]
print(f"Patching: {path}")
src = open(path).read()

target = '            # Handle vision tower embedding layer potential differences'
new_fix = '''            for _vt_sub in ['embeddings', 'encoder', 'post_layernorm']:
                if f'vision_tower.{_vt_sub}.' in new_key and f'vision_tower.vision_model.{_vt_sub}.' not in new_key:
                    new_key = new_key.replace(f'vision_tower.{_vt_sub}.', f'vision_tower.vision_model.{_vt_sub}.')
                    break

'''
if "vision_model.{_vt_sub}." not in src:
    open(path, 'w').write(src.replace(target, new_fix + target))
    print("Vision tower v2 fix applied")
else:
    print("Already patched")

# Set env vars
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["NGROK_TOKEN"] = userdata.get('NGROK_TOKEN')

# Open TCP tunnel
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
tunnel = ngrok.connect(8090, "tcp")
url = tunnel.public_url.replace("tcp://", "")
print(f"Update config.py RUNPOD_SERVER = '{url}'")

# Start server
def run_server():
    with open("/content/server.log", "w") as f:
        subprocess.run(
            ["python", "-m", "lerobot.async_inference.policy_server",
             "--host=0.0.0.0", "--port=8090", "--fps=25"],
            stdout=f, stderr=f
        )

t = threading.Thread(target=run_server, daemon=True)
t.start()
print("Server starting... monitor with:  !tail -f /content/server.log")

Patching: /usr/local/lib/python3.13/dist-packages/lerobot/policies/pi05/modeling_pi05.py
Vision tower v2 fix applied
Update config.py RUNPOD_SERVER = '6.tcp.ngrok.io:21459'
Server starting... monitor with:  !tail -f /content/server.log


In [ ]:
!tail -f /content/server.log

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO 2026-08-22 07:35:22 y_server.py:420 {'fps': 25,
 'host': '0.0.0.0',
 'inference_latency': 0.03333333333333333,
 'obs_queue_timeout': 2,
 'port': 8090}
INFO 2026-08-22 07:35:22 y_server.py:430 PolicyServer started on 0.0.0.0:8090
INFO 2026-08-22 07:35:57 y_server.py:110 Client ipv6:%5B::1%5D:34932 connected and ready
INFO 2026-08-22 07:35:58 y_server.py:136 Receiving policy instructions from ipv6:%5B::1%5D:34932 | Policy type: pi05 | Pretrained name or path: subhodipsaha/pi05_stack_three_cubes_08_19_v2 | Actions per chunk: 50 | Device: cuda
INFO 2026-08-22 07:38:27 ing_pi05.py:449 Enabled gradient checkpointing for PI05Pytorch model
WARNING 2026-08-22 07:38:30 ing_pi05.py:920 Vision embe

In [ ]:
import subprocess
result = subprocess.run(["ps", "aux"], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if 'policy_server' in line:
        print(line)

#And check if port 8080 is listening:

import subprocess
result = subprocess.run(["lsof", "-i", ":8090"], capture_output=True, text=True)
print(result.stdout if result.stdout else "Nothing on port 8090 yet")

Nothing on port 8090 yet


In [ ]:
import subprocess
subprocess.run(["kill", "-9", "6540"])
import time
time.sleep(2)
result = subprocess.run(["lsof", "-i", ":8090"], capture_output=True, text=True)
print(result.stdout if result.stdout else "Port 8090 is free")

Port 8090 is free


In [ ]:
from pyngrok import ngrok
tunnels = ngrok.get_tunnels()
print(f"Active tunnels: {tunnels}")

Active tunnels: []


In [ ]:
import subprocess
result = subprocess.run(["ps", "aux"], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if 'policy_server' in line:
        print(line)

root        6540 38.7  9.2 32759364 5139500 ?    Sl   07:00   3:53 python3 -m lerobot.async_inference.policy_server --host=0.0.0.0 --port=8090 --fps=25
